In [137]:
import pandas as pd
import numpy as np

In [138]:
# Load dataset
df = pd.read_csv("../data/raw/Metro_Manila_Traffic_Incidents_2025.csv")


In [139]:
#Size
df.shape

(1010, 17)

In [140]:
#Data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Incident_ID        1010 non-null   str    
 1   Date               1010 non-null   str    
 2   Time               1010 non-null   str    
 3   City               1010 non-null   str    
 4   Road_Name          1010 non-null   str    
 5   Vehicle_Type       1010 non-null   str    
 6   Accident_Type      1010 non-null   str    
 7   Severity           605 non-null    str    
 8   Weather_Condition  1010 non-null   str    
 9   Road_Condition     614 non-null    str    
 10  Cause              746 non-null    str    
 11  Driver_Age         996 non-null    float64
 12  Driver_Gender      1010 non-null   str    
 13  Injury_Count       845 non-null    float64
 14  Damage_Cost_PHP    1002 non-null   float64
 15  Latitude           1010 non-null   float64
 16  Longitude          1010 non-null   

In [141]:
#Columns names
df.columns

Index(['Incident_ID', 'Date', 'Time', 'City', 'Road_Name', 'Vehicle_Type',
       'Accident_Type', 'Severity', 'Weather_Condition', 'Road_Condition',
       'Cause', 'Driver_Age', 'Driver_Gender', 'Injury_Count',
       'Damage_Cost_PHP', 'Latitude', 'Longitude'],
      dtype='str')

In [142]:
#Clean column names using snake_case and in lowercase

df.columns = df.columns.str.lower()
df.columns

Index(['incident_id', 'date', 'time', 'city', 'road_name', 'vehicle_type',
       'accident_type', 'severity', 'weather_condition', 'road_condition',
       'cause', 'driver_age', 'driver_gender', 'injury_count',
       'damage_cost_php', 'latitude', 'longitude'],
      dtype='str')

In [143]:
# Handle missing values in categorical columns

df = df.fillna({
    "severity": "Unknown",
    "road_condition": "Unknown",
    "cause": "Unknown",
    "injury_count": 0
})

In [144]:
# Removes rows where every column value is identical to another row.
df = df.drop_duplicates()

#Verify
df.duplicated().sum()

np.int64(0)

In [145]:
#Strip whitespace
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include=["object", "string"]):
    df[col] = df[col].str.strip()

#Verify
for col in df.select_dtypes(include=["object", "string"]):
    print(col, df[col].str.contains(r"^\s|\s$", regex=True).sum())




incident_id 0
date 0
time 0
city 0
road_name 0
vehicle_type 0
accident_type 0
severity 0
weather_condition 0
road_condition 0
cause 0
driver_gender 0


In [146]:
# Validate primary key integrity:
# Ensure incident_id has no duplicates and matches total row count
total_rows = df.shape[0]
unique_ids = df["incident_id"].nunique()

print(total_rows, unique_ids)


1000 1000


In [147]:
# Datetime Standardization & Validation

# Combine date and time columns into a single datetime column
# Invalid or improperly formatted values will be converted to NaT
df["datetime"] = pd.to_datetime(
    df["date"] + " " + df["time"],
    errors="coerce"
)
invalid = df[df["datetime"].isna()]

#Verify
df["datetime"].isna().sum()

# Check Min and Max(All should be in range of year 2025)
df["datetime"].min()
df["datetime"].max()

#Drop date and time column
df = df.drop(columns=["date", "time"])
df.columns




Index(['incident_id', 'city', 'road_name', 'vehicle_type', 'accident_type',
       'severity', 'weather_condition', 'road_condition', 'cause',
       'driver_age', 'driver_gender', 'injury_count', 'damage_cost_php',
       'latitude', 'longitude', 'datetime'],
      dtype='str')

In [148]:
#Standardize City names
df["city"] = df["city"].str.strip().str.title()

#Verify
df["city"].value_counts(dropna=False)

city
Pasig          112
Quezon City    107
Manila         100
Makati          93
Las Piñas       59
Navotas         58
Parañaque       52
Malabon         52
Marikina        51
Taguig          49
Valenzuela      48
Pasay           47
Caloocan        46
Mandaluyong     45
San Juan        42
Pateros         39
Name: count, dtype: int64

In [149]:
# Analyze frequency of road_name values to identify inconsistencies and dominant entries
df["road_name"].value_counts(dropna=False)

road_name
Ortigas Ave         111
Roxas Blvd          108
Quezon Ave          105
Taft Ave            105
Commonwealth Ave    103
C5                  101
Aurora Blvd          95
España Blvd          94
EDSA                 91
Katipunan Ave        87
Name: count, dtype: int64

In [150]:
# Analyze frequency of vehicle_type values to identify inconsistencies and dominant entries
df["vehicle_type"].value_counts(dropna=False)

vehicle_type
Truck         171
Taxi          158
UV Express    151
Bus           140
Car           134
Motorcycle    123
Jeepney       123
Name: count, dtype: int64

In [151]:
# Analyze frequency of accident_type values to identify inconsistencies and dominant entries
df["accident_type"].value_counts(dropna=False)

accident_type
Hit and Run    217
Rear-end       208
Collision      193
Side-swipe     192
Pedestrian     190
Name: count, dtype: int64

In [152]:
# Analyze frequency of severity values to identify inconsistencies and dominant entries
df["severity"].value_counts(dropna=False)


severity
Unknown    401
Minor      211
Major      202
Fatal      186
Name: count, dtype: int64

In [153]:
# Analyze frequency of weather_condition values to identify inconsistencies and dominant entries
df["weather_condition"].value_counts(dropna=False)

#Standardize City names
df["weather_condition"] = df["weather_condition"].str.strip().str.title()

#Verify
df["weather_condition"].value_counts(dropna=False)

weather_condition
Clear     344
Rain      324
Cloudy    180
Storm     152
Name: count, dtype: int64

In [154]:
# Analyze frequency of road_condition values to identify inconsistencies and dominant entries
df["road_condition"].value_counts(dropna=False)

road_condition
Unknown     391
Dry         222
Wet         197
Slippery    190
Name: count, dtype: int64

In [155]:
# Analyze frequency of cause values to identify inconsistencies and dominant entries
df["cause"].value_counts(dropna=False)

cause
Unknown               262
Distracted Driving    144
Drunk Driving         130
Mechanical Failure    127
Reckless Driving      126
Traffic Violation     108
Overspeeding          103
Name: count, dtype: int64

In [156]:
# Data Quality Check: Validate and standardize driver_age
#Review distribution and summary statistics
df["driver_age"].describe()

#Check for missing values
df["driver_age"].isna().sum()

#Confirm current data type
df["driver_age"].dtype

#Identify unexpected decimal values (age should be whole number)
df[df["driver_age"] % 1 != 0]

#Detect unrealistic age values (domain validation)
df[df["driver_age"] < 0]
df[df["driver_age"] > 110]

#Convert to nullable integer type after validation
df["driver_age"] = df["driver_age"].astype("Int64")

#Verify: 
df["driver_age"].dtype

df["driver_age"].isna().sum()

np.int64(14)

In [157]:
# Analyze frequency of driver_gender values to identify inconsistencies and dominant entries
df["driver_gender"].value_counts(dropna=False)

#Standardize Gender
df["driver_gender"] = df["driver_gender"].replace({
    "male": "Male",
    "FEMALE": 'Female',
})

#Verify
df["driver_gender"].value_counts(dropna=False)

driver_gender
Female    501
Male      499
Name: count, dtype: int64

In [158]:
# Analyze frequency of injury_count values to identify inconsistencies and dominant entries
df["injury_count"].value_counts(dropna=False)

df["injury_count"].dtype

dtype('float64')

In [ ]:
#Remove negative damage costs

#Convert to Integer
df["damage_cost_php"] = df["damage_cost_php"].astype("Int64")

#Check if Any Negative Numbers Exist
(df["damage_cost_php"] < 0).any()

#Check How Many Negative Values
(df["damage_cost_php"] < 0).sum()

#See those negative rows
df[df["damage_cost_php"] < 0]

#Replacte negative damage cost by NaN
df.loc[df["damage_cost_php"]< 0, "damage_cost_php"] = np.nan

#Verify
df.head

<bound method NDFrame.head of         incident_id         city    road_name vehicle_type accident_type  \
0    MMTI-2025-0001  Mandaluyong   Quezon Ave          Car      Rear-end   
1    MMTI-2025-0002    Parañaque  España Blvd        Truck      Rear-end   
2    MMTI-2025-0003      Navotas           C5          Bus    Side-swipe   
3    MMTI-2025-0004    Parañaque           C5   Motorcycle      Rear-end   
4    MMTI-2025-0005  Mandaluyong           C5      Jeepney      Rear-end   
..              ...          ...          ...          ...           ...   
995  MMTI-2025-0996        Pasay     Taft Ave          Car      Rear-end   
996  MMTI-2025-0997    Parañaque  Ortigas Ave          Bus   Hit and Run   
997  MMTI-2025-0998  Quezon City     Taft Ave         Taxi     Collision   
998  MMTI-2025-0999  Mandaluyong   Quezon Ave      Jeepney     Collision   
999  MMTI-2025-1000        Pasay  Aurora Blvd        Truck    Pedestrian   

    severity weather_condition road_condition            

In [160]:
#Data Quality Check: Validate Geographic Coordinates

#Ensure numeric type (convert if necessary)
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

#Check for missing values
lat_missing = df["latitude"].isna().sum()
lon_missing = df["longitude"].isna().sum()

# Latitude must be between -90 and 90
invalid_lat_global = df[
    (df["latitude"] < -90) | (df["latitude"] > 90)
]

# Longitude must be between -180 and 180
invalid_lon_global = df[
    (df["longitude"] < -180) | (df["longitude"] > 180)
]

# Confirm values fall within Philippines geographic range
# Approximate PH bounds: Latitude (4–21), Longitude (116–127)
invalid_lat_ph = df[
    (df["latitude"] < 4) | (df["latitude"] > 21)
]

invalid_lon_ph = df[
    (df["longitude"] < 116) | (df["longitude"] > 127)
]

# Detect potential swapped coordinates
# (Latitude unusually high for PH or longitude unusually low)
potential_swapped = df[
    (df["latitude"] > 90) | (df["longitude"] < 0)
]

#Detect placeholder coordinates (0,0)
zero_coordinates = df[
    (df["latitude"] == 0) & (df["longitude"] == 0)
]

#Verify
print("Missing Latitude:", lat_missing)
print("Missing Longitude:", lon_missing)
print("Invalid Global Latitude:", len(invalid_lat_global))
print("Invalid Global Longitude:", len(invalid_lon_global))
print("Invalid PH Latitude:", len(invalid_lat_ph))
print("Invalid PH Longitude:", len(invalid_lon_ph))
print("Potential Swapped:", len(potential_swapped))
print("Zero Coordinates:", len(zero_coordinates))


Missing Latitude: 0
Missing Longitude: 0
Invalid Global Latitude: 0
Invalid Global Longitude: 0
Invalid PH Latitude: 0
Invalid PH Longitude: 0
Potential Swapped: 0
Zero Coordinates: 0


In [ ]:
# Feature Engineering

#Get Hour
df["hour"] = df["datetime"].dt.strftime("%#I %p")

#Day of the week
df["day_of_week"] = df["datetime"].dt.day_name()

#Get Month
df["day_of_week"] = df["datetime"].dt.month_name()

#Is weekend
df["is_weekend"] = df["datetime"].dt.weekday >= 5

#Damage Cost Category
df["cost_category"] = pd.cut(
    df["damage_cost_php"],
    bins=[0,30000,70000,100000],
    labels=["Low", "Medium", "High"]
)

# Age Group Category
df["age_group"] = pd.cut(
    df["driver_age"],
    bins = [0,17,25,40,60,float("inf")],
    labels=["Minor","18-25","26-40","41-60","Senior"]
)

df["age_group"] = df["age_group"].cat.add_categories(["Unknown"]).fillna("Unknown")




np.int64(14)

In [162]:
#Save Processed Data
df.to_csv("../data/processed/traffic_incidents_cleaned.csv", index=False)